# Nepal Imports vs Exports — Data Analysis & Machine Learning Forecast

## 1. Introduction

Nepal has run a large and growing trade deficit for decades — it imports far
more than it exports. This project asks one specific question:

> **Can historical Nepal trade patterns be used to predict future imports
> and exports, and which model performs best at doing so?**

To answer that, this notebook:
- Cleans and combines official Nepal trade data from two sources
- Explores the historical trend, trade partners, and product composition
- Engineers time-series features (lags, rolling averages, growth rates)
- Trains and compares four models — a naive baseline, Linear Regression,
  Polynomial Regression, and Random Forest — using a **chronological**
  train/test split (never shuffling years, since future can't leak into past)
- Picks the best-performing model and uses it to forecast the next 5 years


## 2. Data Sources

| Source | What it gives us | Coverage |
|---|---|---|
| **World Bank** — Exports of goods & services (current US$) | Long historical time series | Nepal, 1965–2024 |
| **World Bank** — Imports of goods & services (current US$) | Long historical time series | Nepal, 1965–2024 |
| **Department of Customs, Nepal** — *Nepal Foreign Trade Statistics FY 2079/80* | Recent, detailed, official breakdown | FY 2075/76–2079/80 |
| &nbsp;&nbsp;→ Table 2 | Imports/Exports by fiscal year | 5 years |
| &nbsp;&nbsp;→ Table 5 | Imports/Exports by product (HS chapter) | 97 chapters, 1 year |
| &nbsp;&nbsp;→ Table 6 | Imports/Exports by trading partner country | 170 countries, 1 year |

The World Bank series is what feeds the ML model (it's the only one long
enough to learn a trend from). The Customs tables add real-world detail and
context: *what* Nepal trades and *with whom*, right now.


## Setup — import libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np
import zipfile
import os

# Interactive charts
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)

# A small helper so every chart shares the same clean look (used throughout the notebook)
def style_fig(fig, height=480):
    fig.update_layout(
        template="plotly_white",
        font=dict(size=13),
        title_font=dict(size=18),
        margin=dict(l=60, r=30, t=70, b=50),
        height=height,
    )
    return fig


## 3. Data Loading

Put this notebook in the same folder as your 5 files:
- `API_NE.EXP.GNFS.CD_DS2_en_csv_v2_35140.zip`
- `API_NE.IMP.GNFS.CD_DS2_en_csv_v2_33330.zip`
- `table_2_output-checkpoint.csv`
- `table_5_output-checkpoint.csv`
- `table_6_complete_output.xls`


In [ ]:
# --- Auto-extract the World Bank zip files ---

def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    print(f"Extracted: {zip_path} -> {extract_to}")

extract_zip("API_NE.EXP.GNFS.CD_DS2_en_csv_v2_35140.zip", "wb_exports")
extract_zip("API_NE.IMP.GNFS.CD_DS2_en_csv_v2_33330.zip", "wb_imports")

def find_data_csv(folder):
    for f in os.listdir(folder):
        if f.startswith("API_") and f.endswith(".csv"):
            return os.path.join(folder, f)
    raise FileNotFoundError(f"No data CSV found in {folder}")

wb_exports_path = find_data_csv("wb_exports")
wb_imports_path = find_data_csv("wb_imports")


In [ ]:
# --- Load World Bank data (skip the 4 junk header lines) ---
wb_exports_raw = pd.read_csv(wb_exports_path, skiprows=4)
wb_imports_raw = pd.read_csv(wb_imports_path, skiprows=4)

# --- Load Nepal Customs Department tables ---
table2_raw = pd.read_csv("table_2_output-checkpoint.csv")
table5_raw = pd.read_csv("table_5_output-checkpoint.csv")
table6_raw = pd.read_csv("table_6_complete_output.xls")  # actually a CSV despite the .xls name

print("Table 2 shape:", table2_raw.shape)
print("Table 5 shape:", table5_raw.shape)
print("Table 6 shape:", table6_raw.shape)


## 4. Data Cleaning & Preprocessing

Each dataset was copied out of an official PDF, so the raw files have the
usual problems: stray spaces inside numbers, values landing in the wrong
column, and a couple of tables that had extra unrelated rows mixed in. We
fix each one below.


In [ ]:
# Helper: turn messy PDF-copied numbers into real numbers, e.g. " 9 7,109,521" -> 97109521
def clean_number(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace(",", "").replace(" ", "")
    try:
        return float(x)
    except ValueError:
        return np.nan


### 4a. Table 2 — Yearly trend (5 fiscal years)

In [ ]:
print(table2_raw.columns.tolist())
table2_raw.head(15)


In [ ]:
# Keep only the first 5 rows (the yearly trend table) -- a second, unrelated
# table was pasted below it in the same file, so we drop everything after row 5.
table2_clean = pd.DataFrame({
    "fiscal_year": table2_raw.iloc[0:5, 0].astype(str).str.strip(),
    "exports_npr_000": table2_raw.iloc[0:5].filter(like="Export").iloc[:, -1].apply(clean_number)
                        if len(table2_raw.filter(like="Export").columns) else table2_raw.iloc[0:5, 1].apply(clean_number),
    "imports_npr_000": table2_raw.iloc[0:5].filter(like="Import").iloc[:, -1].apply(clean_number)
                        if len(table2_raw.filter(like="Import").columns) else table2_raw.iloc[0:5, 3].apply(clean_number),
    "trade_balance_npr_000": table2_raw.iloc[0:5].filter(like="Balance").iloc[:, -1].apply(clean_number)
                        if len(table2_raw.filter(like="Balance").columns) else table2_raw.iloc[0:5, 5].apply(clean_number),
})
table2_clean


**Check this against the official numbers before moving on:**

| Fiscal Year | Exports | Imports | Trade Balance |
|---|---|---|---|
| 2075/76 | 97,109,521 | 1,418,535,343 | -1,321,425,822 |
| 2076/77 | 97,709,105 | 1,196,799,053 | -1,099,089,948 |
| 2077/78 | 141,124,080 | 1,539,837,068 | -1,398,712,987 |
| 2078/79 | 200,030,962 | 1,920,448,349 | -1,720,417,388 |
| 2079/80 | 157,140,695 | 1,611,731,770 | -1,454,591,074 |

If a column is off, check `table2_raw.columns.tolist()` above and adjust the
`.iloc[0:5, N]` column positions.


### 4b. Table 5 — Imports/Exports by product (HS chapter)

In [ ]:
print(table5_raw.columns.tolist())
table5_raw.head(10)


In [ ]:
# Some product descriptions got split across two rows when copied from the PDF.
# Fix: forward-fill the description down, then keep only rows that have a real import value.

table5 = table5_raw.copy()

imp_col = [c for c in table5.columns if "Import" in c][0]
exp_col = [c for c in table5.columns if "Export" in c][0]
desc_col = [c for c in table5.columns if "Description" in c][0]
chapter_col = [c for c in table5.columns if "Chapter" in c][0]
revenue_col = next((c for c in table5.columns if "Revenue" in c), None)
balance_col = next((c for c in table5.columns if "Balance" in c), None)

table5[desc_col] = table5[desc_col].ffill()
table5_clean = table5[table5[imp_col].notna()].copy()

table5_clean[imp_col] = table5_clean[imp_col].apply(clean_number)
table5_clean[exp_col] = table5_clean[exp_col].apply(clean_number)

keep_cols = [chapter_col, desc_col, imp_col, exp_col]
rename_map = {chapter_col: "chapter", desc_col: "description",
              imp_col: "imports_npr_000", exp_col: "exports_npr_000"}

if revenue_col:
    table5_clean[revenue_col] = table5_clean[revenue_col].apply(clean_number)
    keep_cols.append(revenue_col)
    rename_map[revenue_col] = "revenue_npr_000"

table5_clean = table5_clean[keep_cols].rename(columns=rename_map)
table5_clean["trade_balance_npr_000"] = table5_clean["exports_npr_000"] - table5_clean["imports_npr_000"]

# Drop the 'Total' summary row -- we don't want it treated as a product
table5_clean = table5_clean[~table5_clean["description"].str.contains("Total", case=False, na=False)]
table5_clean = table5_clean.dropna(subset=["chapter"])

table5_clean.head(10)


### 4c. Table 6 — Imports/Exports by trade partner country

In [ ]:
print(table6_raw.columns.tolist())
table6_raw.head(10)


In [ ]:
# This file also has Table 7 (HS-code line-item detail) accidentally pasted below Table 6.
# Table 6 rows always have a country name; Table 7 rows don't -- we use that to separate them.

country_col = [c for c in table6_raw.columns if "Countr" in c][0]
imp_col6 = [c for c in table6_raw.columns if "Import" in c][0]
exp_col6 = [c for c in table6_raw.columns if "Export" in c][0]

table6 = table6_raw.copy()

def pick_country_name(row):
    for col in table6_raw.columns[:4]:
        val = row[col]
        if pd.notna(val) and not str(val).replace('.', '', 1).isdigit():
            return str(val).strip()
    return np.nan

table6["country"] = table6.apply(pick_country_name, axis=1)
table6[imp_col6] = table6[imp_col6].apply(clean_number)
table6[exp_col6] = table6[exp_col6].apply(clean_number)

table6_clean = table6[table6["country"].notna() & table6[imp_col6].notna()].copy()
table6_clean = table6_clean[~table6_clean["country"].str.contains("Total", case=False, na=False)]

table6_clean = table6_clean[["country", imp_col6, exp_col6]]
table6_clean.columns = ["country", "imports_npr_000", "exports_npr_000"]
table6_clean["total_trade_npr_000"] = table6_clean["imports_npr_000"] + table6_clean["exports_npr_000"]

print("Countries kept:", len(table6_clean))
top_partners = table6_clean.sort_values("total_trade_npr_000", ascending=False).head(15)
top_partners


### 4d. World Bank data — Nepal + South Asian neighbors

In [ ]:
countries_of_interest = {
    "Nepal": "NPL", "India": "IND", "Bangladesh": "BGD",
    "Sri Lanka": "LKA", "Bhutan": "BTN",
}

def clean_world_bank(df, value_name):
    df = df[df["Country Name"].isin(countries_of_interest.keys())].copy()
    year_cols = [c for c in df.columns if c.isdigit()]
    df_long = df.melt(id_vars=["Country Name"], value_vars=year_cols,
                       var_name="year", value_name=value_name)
    df_long["year"] = df_long["year"].astype(int)
    return df_long.dropna(subset=[value_name])

wb_exports_long = clean_world_bank(wb_exports_raw, "exports_usd")
wb_imports_long = clean_world_bank(wb_imports_raw, "imports_usd")

wb_trade = pd.merge(wb_exports_long, wb_imports_long, on=["Country Name", "year"], how="inner")
wb_trade["trade_balance_usd"] = wb_trade["exports_usd"] - wb_trade["imports_usd"]
wb_trade = wb_trade.rename(columns={"Country Name": "country"})

# Nepal-only slice, sorted by year -- this is what feeds the ML model later
nepal_trade = wb_trade[wb_trade["country"] == "Nepal"].sort_values("year").reset_index(drop=True)
print("Years of Nepal data:", nepal_trade["year"].min(), "-", nepal_trade["year"].max(), 
      f"({len(nepal_trade)} years)")
nepal_trade.head()


## 5. Exploratory Data Analysis (EDA)

Before charting, some quick checks: does the data look right, is anything
missing, and what's the overall shape and spread of values.


In [ ]:
print("--- Nepal World Bank trade series: summary stats ---")
print(nepal_trade[["exports_usd", "imports_usd", "trade_balance_usd"]].describe())
print()
print("Missing values:\n", nepal_trade.isna().sum())


In [ ]:
# Sanity check: does our cleaned Table 2 total match the PDF's official figure?
official_2079_80_imports = 1611731770  # from the source PDF, in '000 NPR
ours = table2_clean.loc[table2_clean["fiscal_year"] == "2079/80", "imports_npr_000"].values
print("Official 2079/80 imports:", official_2079_80_imports)
print("Our cleaned value:       ", ours)


In [ ]:
# Sanity check: do Table 6 country totals roughly match the official total imports figure?
print("Sum of Table 6 imports:", table6_clean['imports_npr_000'].sum())
print("Official total:        ", official_2079_80_imports)


### 5a. Distribution of product-level import values

Across the 97 product chapters, is trade spread evenly or dominated by a
handful of categories? A histogram shows the shape of that distribution.


In [ ]:
fig_hist = px.histogram(
    table5_clean, x="imports_npr_000", nbins=40,
    labels={"imports_npr_000": "Imports per Chapter (NPR '000)"},
    title="Distribution of Import Values Across 97 Product Chapters"
)
fig_hist.update_layout(bargap=0.05)
style_fig(fig_hist)
fig_hist.show()


The distribution is heavily right-skewed: most chapters import a small
amount, while a few (like mineral fuels) import an enormous amount. This is
exactly the "trade concentration" pattern we'll see again in the top
commodities chart below.


### 5b. Correlation between product-level trade variables

Do chapters with high imports also tend to generate high customs revenue?
A correlation heatmap across Table 5's numeric columns answers that.


In [ ]:
corr_cols = [c for c in ["imports_npr_000", "exports_npr_000", "trade_balance_npr_000", "revenue_npr_000"]
             if c in table5_clean.columns]
corr_matrix = table5_clean[corr_cols].corr()

fig_heatmap = px.imshow(
    corr_matrix, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Correlation Between Product-Level Trade Variables"
)
style_fig(fig_heatmap, height=450)
fig_heatmap.show()


A strong positive correlation between Imports and Revenue makes sense —
Nepal collects customs duty on imports, so chapters with bigger import
values naturally generate more revenue.


## 6. Data Visualizations

### 6a. Imports vs Exports — historical trend (1965–2024)

In [ ]:
fig1 = px.line(
    nepal_trade, x="year", y=["exports_usd", "imports_usd"],
    labels={"value": "US$", "year": "Year", "variable": "Type"},
    color_discrete_sequence=["#2E8B57", "#C0392B"],
    title="Nepal: Exports vs Imports (1965–2024)"
)
fig1.update_layout(hovermode="x unified", legend_title_text="")
style_fig(fig1)
fig1.show()


### 6b. Trade Balance / Trade Deficit

In [ ]:
fig2 = px.area(
    nepal_trade, x="year", y="trade_balance_usd",
    labels={"trade_balance_usd": "Trade Balance (US$)", "year": "Year"},
    color_discrete_sequence=["#C0392B"],
    title="Nepal: Trade Balance Over Time (Exports − Imports)"
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray")
style_fig(fig2)
fig2.show()


### 6c. Recent Customs Imports vs Exports (official, 5-year detail)

In [ ]:
fig3 = px.bar(
    table2_clean, x="fiscal_year", y=["exports_npr_000", "imports_npr_000"],
    barmode="group",
    color_discrete_sequence=["#2E8B57", "#C0392B"],
    labels={"value": "NPR '000", "fiscal_year": "Fiscal Year", "variable": "Type"},
    title="Nepal Customs: Exports vs Imports by Fiscal Year"
)
fig3.update_layout(legend_title_text="")
style_fig(fig3)
fig3.show()


### 6d. Top Import Commodities

In [ ]:
top_imports = table5_clean.sort_values("imports_npr_000", ascending=False).head(12)

fig4 = px.bar(
    top_imports.sort_values("imports_npr_000"),
    x="imports_npr_000", y="description", orientation="h",
    color_discrete_sequence=["#C0392B"],
    labels={"imports_npr_000": "Imports (NPR '000)", "description": "Commodity"},
    title="Nepal: Top 12 Import Commodities (FY 2079/80)"
)
style_fig(fig4, height=520)
fig4.show()


Composition view — same data, but as a share of total imports:

In [ ]:
top10_imp = table5_clean.nlargest(10, "imports_npr_000")[["description", "imports_npr_000"]]
other_imp_value = table5_clean["imports_npr_000"].sum() - top10_imp["imports_npr_000"].sum()
pie_imports = pd.concat([top10_imp, pd.DataFrame([{"description": "Other (87 chapters)", "imports_npr_000": other_imp_value}])])

top10_exp = table5_clean.nlargest(10, "exports_npr_000")[["description", "exports_npr_000"]]
other_exp_value = table5_clean["exports_npr_000"].sum() - top10_exp["exports_npr_000"].sum()
pie_exports = pd.concat([top10_exp, pd.DataFrame([{"description": "Other (87 chapters)", "exports_npr_000": other_exp_value}])])

fig_pies = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "pie"}]],
                          subplot_titles=("Import Composition", "Export Composition"))
fig_pies.add_trace(go.Pie(labels=pie_imports["description"], values=pie_imports["imports_npr_000"], hole=0.4), 1, 1)
fig_pies.add_trace(go.Pie(labels=pie_exports["description"], values=pie_exports["exports_npr_000"], hole=0.4), 1, 2)
fig_pies.update_layout(title_text="Nepal: Import vs Export Composition by Product (Top 10 + Other)")
style_fig(fig_pies, height=520)
fig_pies.show()


### 6e. Top Trading Partners

In [ ]:
fig5 = px.bar(
    top_partners.sort_values("total_trade_npr_000"),
    x="total_trade_npr_000", y="country", orientation="h",
    color_discrete_sequence=["#34495E"],
    labels={"total_trade_npr_000": "Total Trade (NPR '000)", "country": "Country"},
    title="Nepal: Top 15 Trading Partners (FY 2079/80)"
)
style_fig(fig5, height=520)
fig5.show()


### 6f. South Asian Comparison — Growth Rates (not absolute values)

Comparing raw dollar values isn't fair — India's economy is many times
Nepal's size. Comparing **year-over-year growth rate** instead puts every
country on the same scale.


In [ ]:
wb_trade_sorted = wb_trade.sort_values(["country", "year"]).copy()
wb_trade_sorted["total_trade_usd"] = wb_trade_sorted["exports_usd"] + wb_trade_sorted["imports_usd"]
wb_trade_sorted["trade_growth_pct"] = wb_trade_sorted.groupby("country")["total_trade_usd"].pct_change() * 100

# Growth rates can be extremely noisy in the earliest years (small base values) -- 
# we start the chart from 1990 onward so the trend is actually readable.
growth_plot_df = wb_trade_sorted[wb_trade_sorted["year"] >= 1990]

fig6 = px.line(
    growth_plot_df, x="year", y="trade_growth_pct", color="country",
    labels={"trade_growth_pct": "YoY Total Trade Growth (%)", "year": "Year"},
    title="South Asia: Year-over-Year Total Trade Growth Rate (1990–2024)"
)
fig6.add_hline(y=0, line_dash="dash", line_color="gray")
fig6.update_layout(hovermode="x unified", legend_title_text="")
style_fig(fig6)
fig6.show()


## 7. Feature Engineering

The ML model needs to learn from *patterns in the recent past*, not just
the raw year number. So for every year, we engineer these features from
Nepal's own trade history:

| Feature | What it means |
|---|---|
| `exports_lag1` / `imports_lag1` | Last year's value |
| `exports_lag2` / `imports_lag2` | Value from 2 years ago |
| `exports_lag3` / `imports_lag3` | Value from 3 years ago |
| `exports_roll3` / `imports_roll3` | Average of the previous 3 years (smooths out noise) |
| `trade_balance_lag1` | Last year's trade balance |
| `exports_growth_lag1` / `imports_growth_lag1` | % growth between lag2 and lag1 |
| `year` | The calendar year itself |

**Important:** every feature uses only *past* years relative to the row
being predicted — using this year's own exports to predict this year's
exports would be cheating (data leakage). The first 3 years of the dataset
get dropped, since they don't have 3 full years of history behind them yet.


In [ ]:
ml_df = nepal_trade[["year", "exports_usd", "imports_usd"]].sort_values("year").reset_index(drop=True)

for lag in [1, 2, 3]:
    ml_df[f"exports_lag{lag}"] = ml_df["exports_usd"].shift(lag)
    ml_df[f"imports_lag{lag}"] = ml_df["imports_usd"].shift(lag)

# Rolling 3-year average, using only PAST years (shift(1) first so today's value isn't included)
ml_df["exports_roll3"] = ml_df["exports_usd"].shift(1).rolling(3).mean()
ml_df["imports_roll3"] = ml_df["imports_usd"].shift(1).rolling(3).mean()

# Last year's trade balance
ml_df["trade_balance_lag1"] = ml_df["exports_lag1"] - ml_df["imports_lag1"]

# Growth rate between 2 years ago and last year
ml_df["exports_growth_lag1"] = ml_df["exports_lag1"] / ml_df["exports_lag2"] - 1
ml_df["imports_growth_lag1"] = ml_df["imports_lag1"] / ml_df["imports_lag2"] - 1

ml_df = ml_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

feature_cols = ["year",
                "exports_lag1", "exports_lag2", "exports_lag3", "exports_roll3", "exports_growth_lag1",
                "imports_lag1", "imports_lag2", "imports_lag3", "imports_roll3", "imports_growth_lag1",
                "trade_balance_lag1"]

print(f"Rows available for modeling: {len(ml_df)} (years {ml_df['year'].min()}-{ml_df['year'].max()})")
ml_df.head()


## 8. Machine Learning

**The question we're answering:** can Nepal's own trade history predict its
future imports and exports — and which modeling approach does that best?

We train two separate sets of models — one predicting **Exports**, one
predicting **Imports** — and compare four approaches for each:

1. **Baseline** — naive persistence: "next year = same as last year." Not a
   real model, just a sanity floor. If a real model can't beat this, it's
   not adding value.
2. **Linear Regression** — a straight-line fit across all the engineered features.
3. **Polynomial Regression** *(kept from the earlier version)* — fits a
   curve using Year and Year², since trade growth isn't perfectly linear.
4. **Random Forest** — an ensemble of decision trees; can capture
   non-linear relationships between the lag/rolling features without us
   having to specify the shape by hand.

**Chronological split:** because this is time-series data, we train on the
earliest ~85% of years and test on the most recent ~15% — never shuffling,
since a model should only ever "know" the past when predicting the future.


In [ ]:
split_idx = int(len(ml_df) * 0.85)
train_df = ml_df.iloc[:split_idx]
test_df = ml_df.iloc[split_idx:]

print(f"Train: {train_df['year'].min()}-{train_df['year'].max()} ({len(train_df)} years)")
print(f"Test:  {test_df['year'].min()}-{test_df['year'].max()} ({len(test_df)} years)")

X_train, X_test = train_df[feature_cols], test_df[feature_cols]
y_train_exp, y_test_exp = train_df["exports_usd"], test_df["exports_usd"]
y_train_imp, y_test_imp = train_df["imports_usd"], test_df["imports_usd"]


In [ ]:
results = []          # holds MAE/RMSE/R2 for every model x target, for the comparison table
predictions = {}      # holds test-set predictions for every model x target, for the actual-vs-predicted plots

def evaluate(target_name, model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    results.append({"Target": target_name, "Model": model_name, "MAE": mae, "RMSE": rmse, "R2": r2})
    predictions[(target_name, model_name)] = y_pred
    return mae, rmse, r2


### 8a. Baseline — naive persistence

In [ ]:
# "Predict this year = last year's actual value" -- uses the lag1 feature directly, no fitting needed
baseline_pred_exp = X_test["exports_lag1"].values
baseline_pred_imp = X_test["imports_lag1"].values

evaluate("Exports", "Baseline (persistence)", y_test_exp, baseline_pred_exp)
evaluate("Imports", "Baseline (persistence)", y_test_imp, baseline_pred_imp)
print("Baseline evaluated.")


### 8b. Linear Regression

In [ ]:
lin_exp = LinearRegression().fit(X_train, y_train_exp)
lin_imp = LinearRegression().fit(X_train, y_train_imp)

evaluate("Exports", "Linear Regression", y_test_exp, lin_exp.predict(X_test))
evaluate("Imports", "Linear Regression", y_test_imp, lin_imp.predict(X_test))
print("Linear Regression evaluated.")


### 8c. Polynomial Regression (Year, Year²) — kept from the earlier version

In [ ]:
# Uses only 'year' as the input, fit as a degree-2 curve -- same modeling idea as before,
# now evaluated on the same train/test rows as the other models for a fair comparison.
poly_exp = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(train_df[["year"]], y_train_exp)
poly_imp = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(train_df[["year"]], y_train_imp)

evaluate("Exports", "Polynomial Regression", y_test_exp, poly_exp.predict(test_df[["year"]]))
evaluate("Imports", "Polynomial Regression", y_test_imp, poly_imp.predict(test_df[["year"]]))
print("Polynomial Regression evaluated.")


### 8d. Random Forest Regression

In [ ]:
# max_depth is kept shallow on purpose -- with only ~45 training rows, a deep forest would
# just memorize the training years instead of learning a general pattern (overfitting).
rf_exp = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42).fit(X_train, y_train_exp)
rf_imp = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42).fit(X_train, y_train_imp)

evaluate("Exports", "Random Forest", y_test_exp, rf_exp.predict(X_test))
evaluate("Imports", "Random Forest", y_test_imp, rf_imp.predict(X_test))
print("Random Forest evaluated.")


## 9. Model Evaluation

- **MAE** (Mean Absolute Error) — average size of the error, in US$. Easiest to explain.
- **RMSE** (Root Mean Squared Error) — like MAE, but punishes big misses harder.
- **R²** — how much of the variation in the data the model explains (1.0 = perfect, 0 = no better than guessing the average).


In [ ]:
results_df = pd.DataFrame(results)

print("=== EXPORTS MODEL COMPARISON ===")
display(results_df[results_df["Target"] == "Exports"].sort_values("RMSE").reset_index(drop=True))

print("\n=== IMPORTS MODEL COMPARISON ===")
display(results_df[results_df["Target"] == "Imports"].sort_values("RMSE").reset_index(drop=True))


### Actual vs Predicted — visual model validation

In [ ]:
def actual_vs_predicted_plot(target_name, y_true, years):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=y_true, y=y_true, mode="lines", name="Perfect prediction",
                              line=dict(color="gray", dash="dash")))
    for model_name in ["Baseline (persistence)", "Linear Regression", "Polynomial Regression", "Random Forest"]:
        y_pred = predictions[(target_name, model_name)]
        fig.add_trace(go.Scatter(x=y_true, y=y_pred, mode="markers", name=model_name,
                                  hovertext=[f"Year: {yr}" for yr in years]))
    fig.update_layout(title=f"Actual vs Predicted — {target_name} (Test Years)",
                       xaxis_title="Actual (US$)", yaxis_title="Predicted (US$)")
    style_fig(fig)
    return fig

fig_avp_exp = actual_vs_predicted_plot("Exports", y_test_exp.values, test_df["year"].values)
fig_avp_exp.show()


In [ ]:
fig_avp_imp = actual_vs_predicted_plot("Imports", y_test_imp.values, test_df["year"].values)
fig_avp_imp.show()


**How to read this chart:** the dashed diagonal line is a perfect
prediction. Points sitting close to that line are accurate; points far
above or below it show where a model over- or under-predicted.


## 10. Feature Importance

Which engineered features does the Random Forest actually rely on most when
predicting Exports and Imports?


In [ ]:
def feature_importance_chart(rf_model, target_name):
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf_model.feature_importances_
    }).sort_values("importance", ascending=True)

    fig = px.bar(importance_df, x="importance", y="feature", orientation="h",
                 title=f"Random Forest Feature Importance — {target_name}",
                 labels={"importance": "Importance", "feature": "Feature"})
    style_fig(fig, height=450)
    return fig

fig_fi_exp = feature_importance_chart(rf_exp, "Exports")
fig_fi_exp.show()


In [ ]:
fig_fi_imp = feature_importance_chart(rf_imp, "Imports")
fig_fi_imp.show()


## 11. Future Forecasting — Next 5 Years

**Picking the model:** we use whichever model had the lowest test RMSE for
each target (excluding the baseline, since a flat "same as last year"
prediction isn't a useful forecasting tool by itself — it only exists to
prove the real models are worth using).

**Why this needs care:** our features are *lags* — last year's value, the
year before, etc. To predict 5 years ahead, we can't just plug in "year +
5" once; we have to predict year 1, treat that prediction as if it were
real, use it to build the lag features for year 2, predict year 2, and so
on. This is called **recursive forecasting**.


In [ ]:
candidate_models = ["Linear Regression", "Polynomial Regression", "Random Forest"]

best_exp_model_name = results_df[(results_df["Target"] == "Exports") & (results_df["Model"].isin(candidate_models))] \
                        .sort_values("RMSE").iloc[0]["Model"]
best_imp_model_name = results_df[(results_df["Target"] == "Imports") & (results_df["Model"].isin(candidate_models))] \
                        .sort_values("RMSE").iloc[0]["Model"]

print("Best model for Exports:", best_exp_model_name)
print("Best model for Imports:", best_imp_model_name)


In [ ]:
# Refit the winning model type(s) on ALL available years (not just the training split),
# so the forecast uses every bit of history we have.

def fit_full(model_name, X, y_exports_or_imports):
    if model_name == "Linear Regression":
        return LinearRegression().fit(X[feature_cols], y_exports_or_imports)
    elif model_name == "Polynomial Regression":
        return make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(X[["year"]], y_exports_or_imports)
    elif model_name == "Random Forest":
        return RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42).fit(X[feature_cols], y_exports_or_imports)

final_exp_model = fit_full(best_exp_model_name, ml_df, ml_df["exports_usd"])
final_imp_model = fit_full(best_imp_model_name, ml_df, ml_df["imports_usd"])


In [ ]:
def predict_one_year(model_name, model, year, history):
    """history: dataframe with columns year, exports_usd, imports_usd, sorted by year (actual + predicted so far)"""
    if model_name == "Polynomial Regression":
        return model.predict(pd.DataFrame({"year": [year]}))[0]

    last3 = history.tail(3)
    row = {
        "year": year,
        "exports_lag1": history["exports_usd"].iloc[-1],
        "exports_lag2": history["exports_usd"].iloc[-2],
        "exports_lag3": history["exports_usd"].iloc[-3],
        "exports_roll3": last3["exports_usd"].mean(),
        "exports_growth_lag1": history["exports_usd"].iloc[-1] / history["exports_usd"].iloc[-2] - 1,
        "imports_lag1": history["imports_usd"].iloc[-1],
        "imports_lag2": history["imports_usd"].iloc[-2],
        "imports_lag3": history["imports_usd"].iloc[-3],
        "imports_roll3": last3["imports_usd"].mean(),
        "imports_growth_lag1": history["imports_usd"].iloc[-1] / history["imports_usd"].iloc[-2] - 1,
        "trade_balance_lag1": history["exports_usd"].iloc[-1] - history["imports_usd"].iloc[-1],
    }
    X_future = pd.DataFrame([row])[feature_cols]
    return model.predict(X_future)[0]


In [ ]:
# Recursive forecast loop: predict one year, append it to history, repeat.
history = nepal_trade[["year", "exports_usd", "imports_usd"]].copy()
last_year = history["year"].max()
forecast_rows = []

for step in range(1, 6):
    future_year = last_year + step
    pred_exp = predict_one_year(best_exp_model_name, final_exp_model, future_year, history)
    pred_imp = predict_one_year(best_imp_model_name, final_imp_model, future_year, history)

    forecast_rows.append({"year": future_year, "exports_usd": pred_exp, "imports_usd": pred_imp})
    history = pd.concat([history, pd.DataFrame([forecast_rows[-1]])], ignore_index=True)

forecast_df = pd.DataFrame(forecast_rows)
forecast_df


### Forecast Visualization — historical actuals vs future predictions

In [ ]:
actual_plot = nepal_trade[["year", "exports_usd", "imports_usd"]].copy()
actual_plot["type"] = "Actual"
forecast_plot = forecast_df.copy()
forecast_plot["type"] = "Forecast"
combined = pd.concat([actual_plot, forecast_plot], ignore_index=True)

fig7 = go.Figure()
for col, name, color in [("exports_usd", "Exports", "#2E8B57"), ("imports_usd", "Imports", "#C0392B")]:
    a = combined[combined["type"] == "Actual"]
    f = combined[combined["type"] == "Forecast"]
    fig7.add_trace(go.Scatter(x=a["year"], y=a[col], mode="lines", name=f"{name} (actual)", line=dict(color=color)))
    fig7.add_trace(go.Scatter(x=f["year"], y=f[col], mode="lines+markers", name=f"{name} (forecast)",
                               line=dict(color=color, dash="dash")))

fig7.update_layout(title=f"Nepal Trade Forecast — Next 5 Years "
                          f"(Exports: {best_exp_model_name}, Imports: {best_imp_model_name})",
                    xaxis_title="Year", yaxis_title="US$", hovermode="x unified")
style_fig(fig7)
fig7.show()


## 12. Findings & Conclusion

In [ ]:
# A few numbers to help write your conclusion -- run this last.
total_growth_exp = (nepal_trade["exports_usd"].iloc[-1] / nepal_trade["exports_usd"].iloc[0] - 1) * 100
total_growth_imp = (nepal_trade["imports_usd"].iloc[-1] / nepal_trade["imports_usd"].iloc[0] - 1) * 100

print(f"Nepal's exports grew {total_growth_exp:,.0f}% from {nepal_trade['year'].iloc[0]} to {nepal_trade['year'].iloc[-1]}")
print(f"Nepal's imports grew {total_growth_imp:,.0f}% over the same period")
print()
print(f"Best model for Exports: {best_exp_model_name}")
print(f"Best model for Imports: {best_imp_model_name}")
print()
print(f"5-year forecast (in US$):")
print(forecast_df.to_string(index=False))


**Write your own conclusion here, using the numbers above.** A few
prompts to answer:

- Which model won for each target, and does that match what you'd expect
  given how much data you had (small dataset → simpler models often win)?
- Does the widening trade deficit look likely to continue based on the forecast?
- What are the honest limitations? (Only ~1 country-year of detailed Customs
  data, a 60-year series that's still fairly short for ML, no external
  factors like COVID-19 or policy shocks built into the model.)
- What would you do differently with more time — more years of Customs
  data, a GDP-normalized comparison, exogenous variables like oil prices?


## Export Charts for PowerPoint

Saves every chart as interactive `.html` (open in a browser, or demo live)
and static `.png` (paste directly into slides — needs `pip install kaleido`).


In [ ]:
os.makedirs("chart_exports", exist_ok=True)

figures = {
    "trend_exports_imports": fig1,
    "trade_balance": fig2,
    "recent_5yr": fig3,
    "top_commodities": fig4,
    "commodity_composition_pies": fig_pies,
    "top_partners": fig5,
    "neighbors_growth_comparison": fig6,
    "distribution_histogram": fig_hist,
    "correlation_heatmap": fig_heatmap,
    "actual_vs_predicted_exports": fig_avp_exp,
    "actual_vs_predicted_imports": fig_avp_imp,
    "feature_importance_exports": fig_fi_exp,
    "feature_importance_imports": fig_fi_imp,
    "forecast": fig7,
}

for name, fig in figures.items():
    fig.write_html(f"chart_exports/{name}.html")
print(f"Saved {len(figures)} interactive HTML charts to chart_exports/")


In [ ]:
# Static PNGs for pasting into PowerPoint. If this errors: pip install -U kaleido
try:
    for name, fig in figures.items():
        fig.write_image(f"chart_exports/{name}.png", width=1000, height=600, scale=2)
    print(f"Saved {len(figures)} PNG charts to chart_exports/")
except Exception as e:
    print("PNG export failed -- install kaleido first: pip install -U kaleido")
    print(e)


## 13. Interactive Dashboard

A single-page HTML dashboard, built from your actual computed results above:
KPI summary cards, your key charts (still fully interactive — hover, zoom,
pan), and a forecast-year slider that updates the predicted values live in
the browser. No server needed — just open `dashboard.html` in any browser,
or present it directly during your talk.


In [ ]:
dash_kpis = {
    "latest_year": int(nepal_trade["year"].iloc[-1]),
    "latest_exports": nepal_trade["exports_usd"].iloc[-1],
    "latest_imports": nepal_trade["imports_usd"].iloc[-1],
    "latest_deficit": nepal_trade["trade_balance_usd"].iloc[-1],
    "export_growth_total_pct": total_growth_exp,
    "import_growth_total_pct": total_growth_imp,
    "best_exports_model": best_exp_model_name,
    "best_imports_model": best_imp_model_name,
}
dash_kpis


In [ ]:
import json

# Charts to embed -- keep it to the ones that tell the core story on one page
dashboard_charts = {
    "Historical Trend": fig1,
    "Trade Balance": fig2,
    "Top Trading Partners": fig5,
    "Top Import Commodities": fig4,
    "5-Year Forecast": fig7,
}

chart_html_blocks = ""
for i, (title, fig) in enumerate(dashboard_charts.items()):
    include_js = "cdn" if i == 0 else False  # only load the plotly.js library once
    chart_div = fig.to_html(full_html=False, include_plotlyjs=include_js, div_id=f"chart_{i}")
    chart_html_blocks += f'<div class="chart-card"><h3>{title}</h3>{chart_div}</div>\n'

# Forecast values as JSON, so the slider can read them with plain JavaScript -- no server needed
forecast_json = forecast_df.to_dict(orient="records")
print("Dashboard charts prepared:", list(dashboard_charts.keys()))


In [ ]:
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Nepal Trade Analysis Dashboard</title>
<style>
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #f4f6f8; margin: 0; padding: 30px; color: #222; }}
  h1 {{ text-align: center; color: #1a2b3c; }}
  .subtitle {{ text-align: center; color: #666; margin-bottom: 30px; }}
  .kpi-row {{ display: flex; gap: 16px; flex-wrap: wrap; justify-content: center; margin-bottom: 30px; }}
  .kpi-card {{ background: white; border-radius: 10px; padding: 18px 24px; box-shadow: 0 2px 6px rgba(0,0,0,0.08);
               min-width: 180px; text-align: center; }}
  .kpi-card .label {{ font-size: 13px; color: #888; text-transform: uppercase; letter-spacing: 0.5px; }}
  .kpi-card .value {{ font-size: 24px; font-weight: 700; color: #1a2b3c; margin-top: 6px; }}
  .kpi-card.deficit .value {{ color: #C0392B; }}
  .slider-card {{ background: white; border-radius: 10px; padding: 20px 30px; box-shadow: 0 2px 6px rgba(0,0,0,0.08);
                   max-width: 700px; margin: 0 auto 30px auto; text-align: center; }}
  .slider-card input[type=range] {{ width: 80%; margin-top: 10px; }}
  .slider-values {{ display: flex; justify-content: space-around; margin-top: 14px; font-size: 18px; }}
  .slider-values span.label {{ color: #888; font-size: 13px; display: block; }}
  .chart-card {{ background: white; border-radius: 10px; padding: 20px; margin-bottom: 24px; box-shadow: 0 2px 6px rgba(0,0,0,0.08); }}
  .chart-card h3 {{ margin-top: 0; color: #1a2b3c; }}
</style>
</head>
<body>

<h1>Nepal Trade Analysis Dashboard</h1>
<p class="subtitle">Data through {dash_kpis['latest_year']} · World Bank + Nepal Department of Customs</p>

<div class="kpi-row">
  <div class="kpi-card"><div class="label">Latest Exports</div><div class="value">${dash_kpis['latest_exports']:,.0f}</div></div>
  <div class="kpi-card"><div class="label">Latest Imports</div><div class="value">${dash_kpis['latest_imports']:,.0f}</div></div>
  <div class="kpi-card deficit"><div class="label">Trade Deficit</div><div class="value">${dash_kpis['latest_deficit']:,.0f}</div></div>
  <div class="kpi-card"><div class="label">Export Growth Since 1965</div><div class="value">{dash_kpis['export_growth_total_pct']:,.0f}%</div></div>
  <div class="kpi-card"><div class="label">Best Model (Exports)</div><div class="value" style="font-size:16px;">{dash_kpis['best_exports_model']}</div></div>
  <div class="kpi-card"><div class="label">Best Model (Imports)</div><div class="value" style="font-size:16px;">{dash_kpis['best_imports_model']}</div></div>
</div>

<div class="slider-card">
  <strong>Forecast Explorer</strong> — drag to see the predicted values for each of the next 5 years
  <div>
    <input type="range" id="yearSlider" min="0" max="{len(forecast_json)-1}" value="0" step="1">
  </div>
  <div class="slider-values">
    <div><span class="label">Year</span><span id="valYear">-</span></div>
    <div><span class="label">Predicted Exports</span><span id="valExports">-</span></div>
    <div><span class="label">Predicted Imports</span><span id="valImports">-</span></div>
  </div>
</div>

{chart_html_blocks}

<script>
  const forecastData = {json.dumps(forecast_json)};
  const slider = document.getElementById('yearSlider');
  const valYear = document.getElementById('valYear');
  const valExports = document.getElementById('valExports');
  const valImports = document.getElementById('valImports');

  function updateSlider() {{
    const row = forecastData[slider.value];
    valYear.textContent = row.year;
    valExports.textContent = '$' + Math.round(row.exports_usd).toLocaleString();
    valImports.textContent = '$' + Math.round(row.imports_usd).toLocaleString();
  }}
  slider.addEventListener('input', updateSlider);
  updateSlider();
</script>

</body>
</html>
"""

with open("chart_exports/dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("Dashboard saved to chart_exports/dashboard.html -- open it in any browser.")


## 14. Export Data for the Streamlit Dashboard

Saves the cleaned, processed data as CSVs so the separate `streamlit_app.py`
can load them directly — no need to redo the cleaning steps there.


In [ ]:
os.makedirs("streamlit_data", exist_ok=True)

nepal_trade.to_csv("streamlit_data/nepal_trade.csv", index=False)
wb_trade.to_csv("streamlit_data/wb_trade_neighbors.csv", index=False)
table2_clean.to_csv("streamlit_data/table2_clean.csv", index=False)
table5_clean.to_csv("streamlit_data/table5_clean.csv", index=False)
table6_clean.to_csv("streamlit_data/table6_clean.csv", index=False)
ml_df.to_csv("streamlit_data/ml_df.csv", index=False)
results_df.to_csv("streamlit_data/model_results.csv", index=False)
forecast_df.to_csv("streamlit_data/forecast_default.csv", index=False)

print("Saved processed data to streamlit_data/ -- copy this whole folder next to streamlit_app.py")
